# Lab: Fine-Tuning Pre-Trained Models (Debugging with an LLM)

**Scenario:** As a junior data scientist at a logistics company, I'm adapting a pre-trained language model (DistilBERT) to a specific task via transfer learning. IMDB sentiment data stands in for the company's proprietary data, so the focus is on getting the fine-tuning *pipeline* correct and verifiable.

**Workflow:** review the provided script, list suspected errors, validate them with an LLM, fix and run the code, evaluate honestly, and write up the process.

## Step 1: Obtain the Provided Code

The starter script (copied verbatim, **not executed** because it contains intentional errors):

```python
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
from transformers import DataCollatorWithPadding

# Step 1: Load a Pre-Trained Model and Tokenizer
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labes=2)

# Step 2: Prepare the Fine-Tuning Dataset
dataset = load_dataset("imdb", split="train[:2000]")
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Step 3: Fine-Tune the Pre-Trained Model
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs="3",
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    logging_dir="./logs"
)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    tokenizer=tokenizer,
    data_collator=data_collator
)
trainer.train()

# Step 4: Evaluate and Test the Fine-Tuned Model
test_input = "The movie was absolutely fantastic and I loved it."
test_encoding = tokenizer(test_input, return_tensors="pt")
with torch.no_grad():
    logits = model(test_encoding).logits
predicted_class = torch.argmax(logits, dim=1)
print("Predicted class:", predicted_class.item())
```

**Structure:** (1) load model and tokenizer, (2) load and tokenize data, (3) configure and run `Trainer`, (4) test on a sample sentence.

## Step 2: Identify Errors in the Code

Suspected issues from my own read-through, ordered by how they fail:

| # | Location | Suspected problem | Failure type |
|---|----------|-------------------|--------------|
| 1 | `num_labes=2` | Typo for `num_labels`. Unknown kwargs are absorbed into the config, so nothing raises, and the head only has 2 outputs because 2 is the default. | **Silent** |
| 2 | `num_train_epochs="3"` | String where an int is required. | Crash |
| 3 | `model(test_encoding)` | The tokenizer output is a dict; it must be unpacked with `**`. | Crash |
| 4 | `evaluation_strategy="epoch"` | Renamed to `eval_strategy` in current `transformers`. Also, no `eval_dataset` is provided, so per-epoch evaluation cannot run. | Crash |
| 5 | `train_dataset=tokenized_datasets` | No train/eval split and no `compute_metrics`, so performance cannot be measured. | Logic |
| 6 | `tokenizer=tokenizer` in `Trainer` | Deprecated in favor of `processing_class=`. | Warning / crash |
| 7 | `train[:2000]` | **Hypothesis:** the IMDB train split is sorted by label, so this slice may contain only one class. | **Silent, data bug** |
| 8 | Test cell | No `model.eval()`, no device handling, and a single sentence is weak evidence. | Weak evaluation |

**Hypothesis to test:** items 1-6 are API-level problems. Item 7 is a data problem that would let the script run "successfully" while producing a useless model.

## Step 3: Utilize an LLM for Debugging

### Prompt 1: Broad review
> I have the following Python code intended to fine-tune distilbert-base-uncased on IMDB sentiment data using Hugging Face's Trainer API. Can you review it and identify any errors or misconfigurations? *(full script attached)*

**Response (summary):** `num_train_epochs` must be an int; `model(test_encoding)` needs `**`; `evaluation_strategy` is now `eval_strategy` and needs an `eval_dataset`; `num_labes` is a typo that fails silently; `tokenizer=` is deprecated in favor of `processing_class=`; no `compute_metrics` means no accuracy. Also noted: no `model.eval()` and a single test sentence is thin evidence.

### Prompt 2: Targeted, model loading
> Review only this section for errors. Explain what each argument does and whether a mistake would raise an error or fail silently. `model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labes=2)`

**Response (summary):** `num_labes` is a silent failure: `from_pretrained` forwards unknown kwargs to the config, so no exception is raised. It only works because the default is 2 labels, and it would silently break on a non-binary task. The "newly initialized classifier weights" warning is expected, because the classification head is trained from scratch.

### Prompt 3: Targeted, training configuration
> Check whether these TrainingArguments and the Trainer call are valid for the current transformers version, and whether they will actually let me measure performance.

**Response (summary):** Fix the epoch type; rename `evaluation_strategy` to `eval_strategy` and pass an `eval_dataset`; add `compute_metrics`, otherwise only loss is reported; swap `tokenizer=` for `processing_class=`. Learning rate 2e-5 and batch size 16 are reasonable defaults for DistilBERT.

### Prompt 4: Data-focused (symptom-driven)
> After fixing the syntax errors, my model labeled "The movie was absolutely fantastic and I loved it." as negative. My training data is `load_dataset("imdb", split="train[:2000]")`. What could cause this?

**Response (summary):** The IMDB train split is ordered by label (all negatives first), so `train[:2000]` contains only class 0. The model can reach near-zero loss by always predicting 0. Check with `value_counts()` on the labels. Fix by shuffling *before* slicing and using a stratified split.

**Most useful:** Prompt 4. Prompts 1-3 found API errors that Python's own error messages would eventually surface. Prompt 4 found a problem that produces *no* error message, and it only worked because I gave the LLM the symptom (a wrong prediction) alongside the code.

## Step 4: Modify the Code and Test Corrections

Each change below is tied back to the numbered issues in Step 2.

### 4.1 Import dependencies

In [1]:
import torch
import numpy as np
import transformers
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

set_seed(42)  # reproducible shuffling / initialization of the classification head
print("transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

transformers: 5.20.2
CUDA available: False


### 4.2 Load the pre-trained model and tokenizer
**Fix #1:** `num_labes` → `num_labels`. DistilBERT's body is reused as-is; only the 2-output classification head is new, which is why the "newly initialized weights" warning appears. That warning is expected.

In [2]:
model_checkpoint = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=2,                                   # fixed typo
    id2label={0: "NEGATIVE", 1: "POSITIVE"},        # readable predictions later
    label2id={"NEGATIVE": 0, "POSITIVE": 1},
)
assert model.config.num_labels == 2, "Head does not have 2 outputs"

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### 4.3 Prepare the dataset
**Fix #7 (the silent data bug).** First, prove the hypothesis: look at the label distribution of the original slice.

In [3]:
original_slice = load_dataset("stanfordnlp/imdb", split="train[:2000]")
print("Original slice label counts:", Counter(original_slice["label"]))

Original slice label counts: Counter({0: 2000})


If the output above shows only label `0`, the original slice contains **zero positive reviews**, so a model trained on it can't learn sentiment. The fix is to shuffle the **full** training split first, then select a subset, then do a **stratified** train/evaluation split.

I use 1,000 examples (not 2,000) and a `max_length` of 256 so training finishes in a reasonable time on CPU.

In [4]:
full_train = load_dataset("stanfordnlp/imdb", split="train")
subset = full_train.shuffle(seed=42).select(range(1000))

splits = subset.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")

print("Train label counts:", Counter(splits["train"]["label"]))
print("Eval  label counts:", Counter(splits["test"]["label"]))

Train label counts: Counter({0: 410, 1: 390})
Eval  label counts: Counter({0: 102, 1: 98})


### 4.4 Tokenize
**Fix #8 (efficiency):** truncate to 256 tokens but do **not** pad here. `DataCollatorWithPadding` pads each batch to its longest sequence, which is much cheaper than padding everything to 512.

In [5]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256)

tokenized = splits.map(tokenize_function, batched=True)
tokenized

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 800
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 200
    })
})

### 4.5 Configure training
- **Fix #2:** `num_train_epochs` is now the integer `3`.
- **Fix #4:** `evaluation_strategy` → `eval_strategy`, and an `eval_dataset` is supplied below.
- **Fix #5:** `compute_metrics` added so evaluation reports accuracy, not just loss.
- **Version note:** `logging_dir` from the starter script was removed in `transformers` v5 (it raises a `TypeError`), so it is dropped here. It was only used for TensorBoard logs, which are disabled by `report_to="none"` anyway.

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,                 # int, not "3"
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",              # renamed from evaluation_strategy
    report_to="none",                   # no wandb etc. (logging_dir removed: not supported in transformers v5)
    seed=42,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

### 4.6 Fine-tune
**Fix #5 and #6:** the `Trainer` now receives a separate `train_dataset` and `eval_dataset`, plus `compute_metrics`, and uses `processing_class=` instead of the deprecated `tokenizer=`.

In [7]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    processing_class=tokenizer,         # replaces tokenizer=
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
train_result.metrics

/home/rj/anaconda3/envs/learn-env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.535027,0.765000
2,No log,0.405312,0.835000
3,No log,0.380496,0.830000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/rj/anaconda3/envs/learn-env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'train_runtime': 1301.988,
 'train_samples_per_second': 1.843,
 'train_steps_per_second': 0.115,
 'total_flos': 158960878387200.0,
 'train_loss': 0.4168974304199219,
 'epoch': 3.0}

### 4.7 Evaluate on held-out data
Because the evaluation set is stratified, accuracy is meaningful here. A model that always answers one class would score ~50%, so that is the baseline to beat.

In [8]:
eval_metrics = trainer.evaluate()
print(eval_metrics)

# Per-class accuracy: catches a model that is good at one class and bad at the other
pred_out = trainer.predict(tokenized["test"])
preds = np.argmax(pred_out.predictions, axis=-1)
labels = pred_out.label_ids
for cls, name in [(0, "NEGATIVE"), (1, "POSITIVE")]:
    mask = labels == cls
    print(f"{name}: {(preds[mask] == cls).mean():.2%}  (n={mask.sum()})")
print("Predicted class counts:", Counter(preds.tolist()))

Training Loss,Validation Loss,Epoch,Accuracy
No log,0.380496,3,0.830000


{'eval_loss': 0.3804956078529358, 'eval_accuracy': 0.83}


NEGATIVE: 86.27%  (n=102)
POSITIVE: 79.59%  (n=98)
Predicted class counts: Counter({0: 108, 1: 92})


### 4.8 Test on new inputs
**Fix #3 and #8:** the encoding dict is unpacked with `**`, the model is put in eval mode, inputs are moved to the model's device, and I use several sentences with probabilities rather than a single pass/fail.

The last two sentences are logistics-flavored. They are **out of domain** for a model trained on movie reviews, and are included on purpose to show the limitation (see the report).

In [9]:
model.eval()
device = next(model.parameters()).device

test_inputs = [
    "The movie was absolutely fantastic and I loved it.",
    "A wonderful, moving film with brilliant performances.",
    "Terrible pacing, flat acting, I want my two hours back.",
    "This was a boring mess and a complete waste of time.",
    # Out-of-domain (logistics) examples: expect unreliable behavior
    "The driver arrived early and the package was delivered in perfect condition.",
    "Route 70 closed due to a crash, deliveries delayed by three hours.",
]

for text in test_inputs:
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        logits = model(**enc).logits          # fixed: unpacked with **
    probs = torch.softmax(logits, dim=1)[0]
    pred = int(probs.argmax())
    print(f"{model.config.id2label[pred]:8s} ({probs[pred]:.1%})  <-  {text}")

POSITIVE (92.0%)  <-  The movie was absolutely fantastic and I loved it.
POSITIVE (90.4%)  <-  A wonderful, moving film with brilliant performances.
NEGATIVE (84.7%)  <-  Terrible pacing, flat acting, I want my two hours back.
NEGATIVE (89.4%)  <-  This was a boring mess and a complete waste of time.
POSITIVE (76.0%)  <-  The driver arrived early and the package was delivered in perfect condition.
NEGATIVE (57.9%)  <-  Route 70 closed due to a crash, deliveries delayed by three hours.


### 4.9 Save the fine-tuned model

In [10]:
save_path = "./fine_tuned_model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print("Saved to", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to ./fine_tuned_model


## Step 5: Document Process and Findings

**Errors observed.** Reading the starter script, I flagged three obvious errors: the `num_labes` typo, `num_train_epochs` passed as a string, and `model(test_encoding)` missing the `**` unpacking. The LLM review added API-level problems I hadn't caught: `evaluation_strategy` was renamed `eval_strategy`, no `eval_dataset` was supplied, there was no `compute_metrics`, and `tokenizer=` is deprecated in favor of `processing_class=`. The `num_labes` typo was subtle because it raises no error: unknown kwargs are absorbed into the model config, and the head only has 2 outputs because 2 is the default. The most important issue was in the data rather than the code: the IMDB training split is sorted by label, and I confirmed that `train[:2000]` contains 2,000 negative reviews and zero positive ones (`Counter({0: 2000})`). A model trained on that slice can reach near-zero loss by always predicting "negative", and the technical-lesson notebook, which used the same slicing, labeled an obviously positive sentence as negative. Running the code on my environment (`transformers` 5.20.2) also surfaced version drift that the starter code did not anticipate: `logging_dir` was removed from `TrainingArguments`, and the dataset now has to be loaded as `stanfordnlp/imdb` instead of `imdb`.

**Prompts and what helped.** I started with a broad review prompt, then used narrower prompts on the model-loading section and the `TrainingArguments`/`Trainer` call. The narrow prompts gave more precise explanations, especially about why each error fails loudly or silently. The most useful prompt was symptom-driven: it included the wrong prediction and the dataset line, and it led directly to the label-ordering diagnosis, which a purely syntactic review would not have found. I verified that diagnosis by printing the label counts instead of trusting the explanation.

**Changes and results.** I fixed the typo, changed epochs to an integer, renamed `eval_strategy`, dropped `logging_dir`, unpacked the test encoding with `**`, and switched to `processing_class`. I shuffled the full dataset before selecting 1,000 examples, used a stratified 80/20 split (train: 410 negative / 390 positive; eval: 102 / 98), added a `compute_metrics` function, and moved padding to `DataCollatorWithPadding`. After 3 epochs the model reached **83% accuracy** on the 200 held-out examples (86.3% on negative, 79.6% on positive, predicted class counts 108/92), compared with a 51% baseline for always predicting the majority class. It classified four new movie-review sentences correctly with 85-92% confidence. **Remaining limits:** the evaluation set is small, so the accuracy estimate is noisy, and 83% is solid but not strong for a task where larger models score in the 90s. IMDB is a stand-in for the company's proprietary data. On my two out-of-domain logistics sentences, the model labeled a delivery-praise sentence positive (76%) and a road-closure sentence negative with only 57.9% confidence. Those labels look plausible, but only because sentiment happens to track them: the model cannot distinguish "delayed" from "on time", which is a different task. A real deployment would need labeled logistics data and a target defined around it, such as delay classification. **On LLM-assisted debugging:** LLMs are very good at API-level and version-drift errors, but they review code as text and cannot see your data, and their knowledge of library versions can be out of date (both errors I hit while running the corrected notebook, the dataset ID and logging_dir, came from version changes). Treating their answers as hypotheses and verifying each by running code or inspecting the data is what made the process reliable.
